### Prepare data on Colab

To load data from Google Drive, we first need to mount it to this Colab environment. This will allow you to access your files directly from the notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import os

def normalize_id_backward(raw_id):

    try:
        parts = raw_id.split('_')
        if len(parts) < 3:
            return None

        end_sec = int(float(parts[-1]))
        start_sec = int(float(parts[-2]))
        base_id = "_".join(parts[:-2])

        return f"{base_id}_{start_sec}_{end_sec}"
    except Exception:
        return None

def colab_align_pipeline():
    print("🌊 1. 正在从 Colab 读取三大独立数据湖...")

    # 读取你上传的三个文件
    with open('/content/drive/MyDrive/RAG_Project/videoa11y_datalake.json', 'r', encoding='utf-8') as f:
        a11y_lake = json.load(f)

    valor_lake = {}
    if os.path.exists('/content/drive/MyDrive/RAG_Project/valor_datalake.json'):
        with open('/content/drive/MyDrive/RAG_Project/valor_datalake.json', 'r', encoding='utf-8') as f:
            valor_lake = json.load(f)

    vatex_lake = {}
    if os.path.exists('/content/drive/MyDrive/RAG_Project/vatex_datalake.json'):
        with open('/content/drive/MyDrive/RAG_Project/vatex_datalake.json', 'r', encoding='utf-8') as f:
            vatex_lake = json.load(f)

    # 2. 用反向切分逻辑建立索引映射
    normalized_valor = {normalize_id_backward(k): v for k, v in valor_lake.items() if normalize_id_backward(k)}
    normalized_vatex = {normalize_id_backward(k): v for k, v in vatex_lake.items() if normalize_id_backward(k)}

    master_english = []
    master_chinese = []

    print("🧬 2. 开始进行精准对齐与重组...")

    for a11y_raw_id, a11y_data in a11y_lake.items():
        norm_id = normalize_id_backward(a11y_raw_id)
        if not norm_id:
            continue

        golden_en = a11y_data["Golden"]
        category = a11y_data["Category"]

        draft_en = None
        draft_zh = None
        source = None

        # 优先匹配 VALOR 库
        if norm_id in normalized_valor:
            draft_en = normalized_valor[norm_id]
            source = "VALOR-32K"
        # 其次匹配 VATEX 库
        elif norm_id in normalized_vatex:
            draft_en = normalized_vatex[norm_id]["draft_en"]
            draft_zh = normalized_vatex[norm_id]["draft_zh"]
            source = "VATEX"

        # 只要成功配对到了英文 Draft，就写入英文主表
        if draft_en:
            # 严格遵循你要求的 5 个 Key 格式
            master_english.append({
                "video_id": a11y_raw_id,  # 保留原始带小数点的真实ID，方便后续追溯视频
                "source": source,
                "draft": draft_en,
                "golden": golden_en,
                "category": category
            })

            # 如果同时存在中文 Draft，写入中文主表
            if draft_zh:
                master_chinese.append({
                    "video_id": a11y_raw_id,
                    "source": source,
                    "draft": draft_zh,
                    # 中文黄金描述暂时放英文，后续可以用 Colab 批量调用大模型翻译
                    "golden": f"TODO_TRANSLATE: {golden_en}",
                    "category": category
                })

    print("\n📊 Colab 纯内存匹配战报:")
    print(f"   - 🇬🇧 英文版 Master 数据集 (已完工): {len(master_english)} 条")
    print(f"   - 🇨🇳 中文版 Master 数据集 (待翻译): {len(master_chinese)} 条")

    # 3. 导出保存到 Colab 运行环境中
    with open('master_english.json', 'w', encoding='utf-8') as f:
        json.dump(master_english, f, ensure_ascii=False, indent=4)

    with open('master_chinese.json', 'w', encoding='utf-8') as f:
        json.dump(master_chinese, f, ensure_ascii=False, indent=4)

    print("\n💾 成功！请在 Colab 左侧刷新并下载 master_english.json 和 master_chinese.json！")

if __name__ == "__main__":
    colab_align_pipeline()

🌊 1. 正在从 Colab 读取三大独立数据湖...
🧬 2. 开始进行精准对齐与重组...

📊 Colab 纯内存匹配战报:
   - 🇬🇧 英文版 Master 数据集 (已完工): 38400 条
   - 🇨🇳 中文版 Master 数据集 (待翻译): 8363 条

💾 成功！请在 Colab 左侧刷新并下载 master_english.json 和 master_chinese.json！


In [ ]:
!pip install transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00


In [ ]:
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm

# ==================== 终极超频配置 ====================
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # 零点几B的超级小金刚！
INPUT_FILE = "/content/drive/MyDrive/RAG_Project/master_chinese.json"
OUTPUT_FILE = "master_chinese_v1.json"
BATCH_SIZE = 256                          # 核心：直接开启 256 倍超级并发！
# ====================================================

def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    indices_to_translate = [i for i, item in enumerate(dataset) if item["golden"].startswith("TODO_TRANSLATE:")]
    total_tasks = len(indices_to_translate)
    print(f"📊 过滤完成，总共需要翻译 {total_tasks} 条数据...")

    if total_tasks == 0:
        print("🎉 没有需要翻译的数据，脚本退出。")
        return

    print(f"🤖 正在秒载入 {MODEL_NAME} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 纯原生 Float16 加载，小模型不需要量化减速
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    print(f"🚀 正在释放 {BATCH_SIZE} 倍暴力并发，这波速度会飞起来...")

    # 手动分批
    for i in tqdm(range(0, total_tasks, BATCH_SIZE), desc="翻译总进度"):
        batch_indices = indices_to_translate[i:i+BATCH_SIZE]
        batch_prompts = []

        for idx in batch_indices:
            english_text = dataset[idx]["golden"].replace("TODO_TRANSLATE: ", "").strip()
            # 极简 Prompt 压榨模型思考时间
            messages = [{"role": "user", "content": f"将英文直翻成中文：\n{english_text}"}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            batch_prompts.append(prompt)

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=128, # 限制最大输出
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id
            )

        for out_idx, idx in enumerate(batch_indices):
            input_len = inputs.input_ids[out_idx].shape[0]
            gen_text = tokenizer.decode(generated_ids[out_idx][input_len:], skip_special_tokens=True).strip()
            dataset[idx]["golden"] = gen_text

        # 写入文件
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, ensure_ascii=False, indent=4)

    print(f"\n🎉 闪电战结束！全量数据已保存为 {OUTPUT_FILE}！")

if __name__ == "__main__":
    main()

📊 过滤完成，总共需要翻译 8363 条数据...
🤖 正在秒载入 Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🚀 正在释放 256 倍暴力并发，这波速度会飞起来...


翻译总进度:   0%|          | 0/33 [00:00<?, ?it/s]


🎉 闪电战结束！全量数据已保存为 master_chinese_v1.json！


In [2]:
import json
import random

def split_to_three_parts(input_file, prefix):
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    # 过滤掉坏数据
    dataset = [item for item in dataset if item["draft"] and item["golden"] and not item["golden"].startswith("TODO")]

    # 固定随机种子
    random.seed(42)
    random.shuffle(dataset)

    total = len(dataset)
    train_end = int(total * 0.8)
    val_end = int(total * 0.9)

    train_set = dataset[:train_end]
    val_set = dataset[train_end:val_end]
    test_set = dataset[val_end:]

    # 保存为独立的 JSON
    with open(f"{prefix}_train.json", 'w', encoding='utf-8') as f:
        json.dump(train_set, f, ensure_ascii=False, indent=4)
    with open(f"{prefix}_val.json", 'w', encoding='utf-8') as f:
        json.dump(val_set, f, ensure_ascii=False, indent=4)
    with open(f"{prefix}_test.json", 'w', encoding='utf-8') as f:
        json.dump(test_set, f, ensure_ascii=False, indent=4)

    print(f"📊 {prefix.upper()} 数据集切分完毕：")
    print(f"   - 🏛️ Train (将灌入向量库): {len(train_set)} 条")
    print(f"   - ⚖️ Val (用于调整RAG参数): {len(val_set)} 条")
    print(f"   - 📝 Test (最终期末考试): {len(test_set)} 条\n")

# 执行英文切分
split_to_three_parts("/content/drive/MyDrive/RAG_Project/master_english.json", "en")

# 如果你的中文也翻译完了，把下面这行的注释解开
split_to_three_parts("/content/drive/MyDrive/RAG_Project/master_chinese_v1.json", "zh")

📊 EN 数据集切分完毕：
   - 🏛️ Train (将灌入向量库): 30720 条
   - ⚖️ Val (用于调整RAG参数): 3840 条
   - 📝 Test (最终期末考试): 3840 条

📊 ZH 数据集切分完毕：
   - 🏛️ Train (将灌入向量库): 6690 条
   - ⚖️ Val (用于调整RAG参数): 836 条
   - 📝 Test (最终期末考试): 837 条



In [3]:
# 1. 安装向量数据库客户端以及通用的向量模型库
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [4]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions
from tqdm.notebook import tqdm


# 3. 指定向量库在网盘中的持久化路径
CHROMA_DB_PATH = "/content/drive/MyDrive/RAG_Project"
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# 4. 🔥 核心：分别为中英文配置最专业的 Embedding “翻译官”
print("🤖 正在配置中英双语专属 Embedding 模型...")
# 英文使用 ChromaDB 默认的优秀英文模型
en_embedding_fn = embedding_functions.DefaultEmbeddingFunction()
# 中文使用高度优化的 BAAI 顶尖中文小型向量模型（会自动下载，只需几秒钟）
zh_embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-zh-v1.5")

# 5. 创建两个独立的集合（Collection）
collection_en = chroma_client.get_or_create_collection(name="en_set", embedding_function=en_embedding_fn)
collection_zh = chroma_client.get_or_create_collection(name="zh_set", embedding_function=zh_embedding_fn)

BATCH_SIZE = 1000

# ==================== ⚡ 核心函数：批量灌库 ====================
def import_to_collection(json_file, collection, lang_label):
    if not os.path.exists(json_file):
        print(f"⚠️ 跳过：未找到 {json_file}，请确保该语言的数据已准备就绪。")
        return

    with open(json_file, 'r', encoding='utf-8') as f:
        train_data = json.load(f)

    print(f"📥 成功读取 {len(train_data)} 条 【{lang_label}】 训练数据，开始灌库...")

    for i in tqdm(range(0, len(train_data), BATCH_SIZE), desc=f"{lang_label} 灌库进度"):
        batch = train_data[i:i+BATCH_SIZE]

        documents = []
        metadatas = []
        ids = []

        for item in batch:
            # 拿着未来的新 Draft 去检索历史旧 Draft
            documents.append(item["draft"])

            # 把对应的黄金无障碍描述、分类、数据源塞进元数据
            metadatas.append({
                "golden": item["golden"],
                "category": item["category"],
                "source": item["source"]
            })
            # 保证 ID 的唯一性，中文 ID 加上 _zh 后缀防止和英文冲突
            suffix = "_zh" if lang_label == "中文" else ""
            ids.append(f"{item['video_id']}{suffix}")

        # 执行批量插入
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    print(f"✅ 【{lang_label}】知识库灌录成功！\n")

# ==================== 🚀 执行双语灌库流水线 ====================
# 1. 灌入英文库（3万多条历史黄金范例）
import_to_collection("/content/drive/MyDrive/RAG_Project/en_train.json", collection_en, "英文")

# 2. 灌入中文库（8千多条刚刚由 Qwen 翻译好的中文黄金范例）
import_to_collection("/content/drive/MyDrive/RAG_Project/zh_train.json", collection_zh, "中文")

print("🎉 史诗级大捷！中英双语无障碍向量数据库已全量持久化在您的谷歌云盘中！")

🤖 正在配置中英双语专属 Embedding 模型...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

英文 灌库进度:   0%|          | 0/31 [00:00<?, ?it/s]


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 0.00/79.3M [00:00<?, ?iB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 53.0k/79.3M [00:00<03:03, 453kiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 289k/79.3M [00:00<01:01, 1.35MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   1%|▏         | 1.13M/79.3M [00:00<00:20, 4.04MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   5%|▍         | 3.81M/79.3M [00:00<00:06, 11.5MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   9%|▉         | 7.11M/79.3M [00:00<00:04, 17.5MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  13%|█▎        | 10.6M/79.3M [00:00<00:03, 21.6MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  18%|█▊        | 13.9M/79.3M [00:00<00:02, 23.8MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  22%|██▏       | 17.

✅ 【英文】知识库灌录成功！

📥 成功读取 6690 条 【中文】 训练数据，开始灌库...


中文 灌库进度:   0%|          | 0/7 [00:00<?, ?it/s]

✅ 【中文】知识库灌录成功！

🎉 史诗级大捷！中英双语无障碍向量数据库已全量持久化在您的谷歌云盘中！


In [5]:
import shutil
from google.colab import files
import os

# 1. 向量数据库在你网盘里的路径
CHROMA_DB_PATH = "/content/drive/MyDrive/RAG_Project"
ZIP_NAME = "chroma_db"

if os.path.exists(CHROMA_DB_PATH):
    print("📦 正在将中英双语向量数据库打包压缩，请稍候...")
    # 使用 shutil 自动将网盘里的文件夹压缩为当前目录下的 zip 文件
    shutil.make_archive(ZIP_NAME, 'zip', CHROMA_DB_PATH)
    print("✅ 压缩完成！")

    print("\n📥 正在触发浏览器自动下载...")
    print("⚠️ 提示：如果浏览器弹出了“允许下载多个文件”的提示，请务必点击【允许】！")

    # 触发 Colab 浏览器下载
    files.download(f"{ZIP_NAME}.zip")
else:
    print(f"❌ 错误：没找到路径 {CHROMA_DB_PATH}，请确保你已经运行了挂载云盘和灌库的单元格！")

📦 正在将中英双语向量数据库打包压缩，请稍候...
✅ 压缩完成！

📥 正在触发浏览器自动下载...
⚠️ 提示：如果浏览器弹出了“允许下载多个文件”的提示，请务必点击【允许】！


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>